Feedforward Neural Network (FNN) — Wine Quality Prediction

In [1]:
# CELL 1 — Imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
# CELL 2 — Load dataset (UCI Wine Quality — red wine)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=";")
print(df.shape)
df.head()

(1599, 12)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [3]:
# CELL 3 — Feature/target setup
# Predict quality score (0-10) as a regression problem
feature_cols = [c for c in df.columns if c != "quality"]
X = df[feature_cols].values.astype(np.float32)
y = df["quality"].values.astype(np.float32)

print("Features:", feature_cols)
print("Quality range:", y.min(), "-", y.max())

Features: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']
Quality range: 3.0 - 8.0


In [4]:
# CELL 4 — Train/test split, scaling, tensors
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

print(X_train_t.shape, y_train_t.shape)

torch.Size([1279, 11]) torch.Size([1279, 1])


In [5]:
# CELL 5 — Model definition (plain feedforward network)
class WineQualityFNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)  # linear output — quality is a continuous score
        )

    def forward(self, x):
        return self.net(x)

model = WineQualityFNN(input_dim=X_train_t.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
print(model)

WineQualityFNN(
  (net): Sequential(
    (0): Linear(in_features=11, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [6]:
# CELL 6 — Training loop
epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    preds = model(X_train_t)
    loss = criterion(preds, y_train_t)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} — Loss: {loss.item():.4f}")

Epoch 20/200 — Loss: 3.1637
Epoch 40/200 — Loss: 1.4810
Epoch 60/200 — Loss: 0.8513
Epoch 80/200 — Loss: 0.6148
Epoch 100/200 — Loss: 0.4678
Epoch 120/200 — Loss: 0.4070
Epoch 140/200 — Loss: 0.3817
Epoch 160/200 — Loss: 0.3647
Epoch 180/200 — Loss: 0.3493
Epoch 200/200 — Loss: 0.3352


In [7]:
# CELL 7 — Evaluation
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t)
    mae = torch.mean(torch.abs(test_preds - y_test_t)).item()
    print(f"Test MAE: {mae:.3f} quality points")

for i in range(5):
    print(f"Predicted: {test_preds[i].item():.2f} | Actual: {y_test_t[i].item():.1f}")

Test MAE: 0.464 quality points
Predicted: 5.63 | Actual: 6.0
Predicted: 5.29 | Actual: 5.0
Predicted: 5.42 | Actual: 6.0
Predicted: 5.55 | Actual: 5.0
Predicted: 5.90 | Actual: 6.0


In [8]:
# CELL 8 — Feature importance sanity check (simple permutation test)
def permutation_importance(model, X, y, feature_idx):
    model.eval()
    with torch.no_grad():
        baseline_preds = model(X)
        baseline_mae = torch.mean(torch.abs(baseline_preds - y)).item()

        X_permuted = X.clone()
        perm_indices = torch.randperm(X.shape[0])
        X_permuted[:, feature_idx] = X_permuted[perm_indices, feature_idx]

        permuted_preds = model(X_permuted)
        permuted_mae = torch.mean(torch.abs(permuted_preds - y)).item()

    return permuted_mae - baseline_mae

importances = []
for idx, name in enumerate(feature_cols):
    imp = permutation_importance(model, X_test_t, y_test_t, idx)
    importances.append((name, imp))

importances.sort(key=lambda x: -x[1])
print("Feature importance (MAE increase when shuffled):")
for name, imp in importances:
    print(f"  {name}: {imp:.4f}")

Feature importance (MAE increase when shuffled):
  alcohol: 0.1330
  sulphates: 0.0710
  volatile acidity: 0.0623
  total sulfur dioxide: 0.0545
  density: 0.0343
  free sulfur dioxide: 0.0308
  pH: 0.0297
  fixed acidity: 0.0295
  chlorides: 0.0099
  citric acid: 0.0090
  residual sugar: 0.0074
